<a href="https://colab.research.google.com/github/jazaineam1/BigData2026/blob/main/Cuadernos/7_Elasticsearch_BM25_Compras_Claras.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# S07 · Elasticsearch Search Lab

**Dos recursos y nada más:** la presentación interactiva enseña, pregunta y registra S07 Live; este cuaderno automatiza con Python.

**Propósito.** Construir un buscador textual que puedas explicar y repetir: Console primero, Python después.

**Regla de trabajo:** antes de ejecutar una celda, debes poder explicar qué operación de Elasticsearch automatiza y qué evidencia esperas ver.


## A · Qué vas a construir

Hoy construirás un buscador de texto con Elasticsearch. Antes de programar necesitas entender cinco ideas:

1. **qué información vas a buscar**;
2. **cómo se representa un documento**;
3. **cómo Elasticsearch prepara el texto**;
4. **cómo le haces una solicitud al motor**;
5. **cómo compruebas si los resultados son útiles**.

No asumimos que ya sabes qué es una API, un mapping, un analyzer, un token o BM25. Cada término se define antes de usarlo.

La secuencia de trabajo será:

**presentación → Console de Elastic → laboratorio → Python/Colab**.

Python llega después de entender qué operación está automatizando.

## A1. API y Console desde cero

### ¿Qué es una API?

Una **API** es una forma acordada de pedirle una operación a otro sistema. Tú envías una **solicitud** y el sistema devuelve una **respuesta**.

En Elasticsearch una solicitud suele tener:

- **método HTTP**: una palabra como `GET`, `POST` o `PUT`;
- **ruta**: la operación que quieres usar, por ejemplo `/_analyze`;
- **cuerpo JSON**: datos que acompañan la solicitud;
- **respuesta JSON**: resultado que devuelve Elasticsearch.

### ¿Qué significa POST?

`POST` es un **método HTTP**. En S07 lo usaremos cuando queremos enviar un cuerpo JSON a una operación.

Ejemplo:

```text
POST /_analyze
{
  "analyzer": "standard",
  "text": "Quick brown fox"
}
```

Lee la solicitud así:

- `POST` → método;
- `/_analyze` → ruta de la API;
- `analyzer` y `text` → datos del cuerpo JSON.

> La Analyze API también acepta GET. En esta clase usamos POST porque deja muy visible la estructura **método + ruta + cuerpo**.

### ¿Cómo abro Console en Elastic?

1. Entra a tu proyecto de Elastic.
2. Abre el menú de navegación o usa el buscador global.
3. Busca **Dev Tools**.
4. Entra a **Console**.
5. Pega la solicitud en el editor izquierdo.
6. Selecciona la solicitud y pulsa el botón ▶.
7. Lee la respuesta JSON en el panel derecho.

En algunas vistas del proyecto también aparece una **Persistent Console** en la parte inferior. Sirve para las mismas solicitudes de Elasticsearch.

**ELASTIC CONSOLE · NO SE EJECUTA EN COLAB.**

## A2. Glosario operativo con ejemplos mínimos

No memorices estas palabras aisladas. Cada una responde: **qué es → ejemplo → cómo la observo**.

| Concepto | Qué es | Ejemplo simple | Cómo lo observas |
|---|---|---|---|
| documento | una entidad representada como JSON | una noticia | `_source` |
| campo | una propiedad del documento | `titulo`, `premium` | clave del JSON |
| índice | colección consultable de documentos | `noticias-2026` | Index Management / `count()` |
| corpus | conjunto de documentos sobre los que buscamos o evaluamos | 1.994 procesos | número de documentos preparados |
| mapping | tipos y reglas de los campos | `titulo: text` | `GET indice/_mapping` |
| tokenizer | componente que parte caracteres en unidades | “Quick brown fox” → 3 | `_analyze` |
| token | cada unidad producida | `quick`, `brown`, `fox` | objetos de `tokens[]` |
| analyzer | tokenizer + filtros de análisis | `spanish` | `/_analyze` |
| query | regla de búsqueda | `match` | cuerpo de `/_search` |
| hit | un documento devuelto como resultado | resultado #1 | `hits.hits[]` |
| `_score` | señal numérica para ordenar resultados | 8.21 | cada resultado |

### Token no significa siempre “palabra”

```text
"Quick brown fox"
→ [quick] [brown] [fox]
→ 3 tokens
```

Pero:

```text
"New York"
standard → [new] [york]
keyword  → [new york]
```

Por eso **no existe un conteo universal de tokens** independiente del tokenizer/analyzer.

Cada token real de Elasticsearch puede incluir `position`, `start_offset` y `end_offset`. Compruébalo con `/_analyze`.

## A2. Patrones de búsqueda que aparecen en industria

| Producto | Necesidad textual | Campos que puntúan | Filtros estructurados | Hipótesis de ranking |
|---|---|---|---|---|
| E-commerce | zapatos de senderismo impermeables | `nombre^3`, `descripcion` | talla, stock, precio | el nombre expresa mejor intención |
| Empleo | data scientist NLP | `cargo^3`, `skills^2`, `descripcion` | país, remoto | cargo y skills concentran evidencia |
| Noticias | sobrecostos contratación | `titulo^4`, `subtitulo^2`, `cuerpo` | premium, fecha | el título indica centralidad temática |
| Soporte | error de autenticación token | `titulo^2`, `problema`, `solucion` | producto, versión | recuperar solución correcta primero |

La regla reusable: **necesidad textual → campos que puntúan → filtros → hipótesis → evaluación.**

# B · Preparar Python y el corpus

### ¿Qué es corpus?

En esta sesión llamaremos **corpus** al conjunto de documentos sobre el que vamos a buscar.

Ejemplos:

- 10.000 noticias → corpus de noticias;
- 1.994 procesos contractuales → corpus contractual;
- 50.000 vacantes → corpus de empleo.

Aquí cargamos el corpus que viene de S06 y lo dejamos listo para automatizar más adelante.

**Importante:** todavía no estamos “ejecutando BM25 en Python”. Solo preparamos datos y el cliente de Elasticsearch.

**Debe aparecer:** filas fuente, procesos únicos y una muestra.

**Si falla:** revisa conexión de Colab o que el archivo remoto exista.

In [ ]:
!pip -q install elasticsearch pandas

In [ ]:
import json
import re
from pathlib import Path

import pandas as pd

URL_S06 = 'https://raw.githubusercontent.com/jazaineam1/BigData2026/main/Datos/s06_contexto_relacional.csv'
corpus_raw = pd.read_csv(URL_S06)

columnas = ['id_proceso','nombre_proceso','descripcion','entidad','tipo_registro','url_secop']
for col in columnas:
    if col not in corpus_raw.columns:
        corpus_raw[col] = ''

corpus = (corpus_raw[columnas]
          .fillna('')
          .drop_duplicates('id_proceso')
          .reset_index(drop=True))

corpus['texto_busqueda'] = (
    corpus['nombre_proceso'].astype(str)
    + ' '
    + corpus['descripcion'].astype(str)
)

print('Filas fuente:', len(corpus_raw))
print('Documentos del corpus después de quitar duplicados:', len(corpus))
display(corpus.head(3))


## B1. Cómo leer el código anterior

- `pd.read_csv(URL_S06)` carga los datos producidos en S06.
- `drop_duplicates('id_proceso')` evita indexar dos veces el mismo proceso.
- `texto_busqueda` junta nombre y descripción para una comparación literal sencilla.
- `len(corpus)` es el número de documentos que prepararemos para Elasticsearch.

**Evidencia esperada:** cerca de 2.109 filas fuente y 1.994 procesos únicos.

**Corpus no significa “texto gigante”.** Es simplemente el conjunto de documentos con el que estamos trabajando.

# C · Línea base: una búsqueda literal en Pandas

Antes de Elasticsearch usa algo que ya conoces de Python.

`str.contains()` **no es Elasticsearch**. Es una operación de Pandas que pregunta si una cadena/patrón aparece dentro de cada texto.

Ejemplo:

```python
df["descripcion"].str.contains(
    "mantenimiento",
    case=False,
    na=False,
    regex=False
)
```

Devuelve `True/False` por fila. Eso sirve para filtrar, pero por sí solo no produce un ranking de relevancia.

In [ ]:
consulta = 'mantenimiento aeronaves'
mask = corpus['texto_busqueda'].str.lower().str.contains('mantenimiento', na=False)
print('Coincidencias literales con mantenimiento:', int(mask.sum()))
display(corpus.loc[mask, ['id_proceso','nombre_proceso','entidad']].head(10))

## C1. Entonces, ¿qué es BM25?

**BM25 no es un paquete que debas instalar para usar Elasticsearch.**

BM25 es un **modelo de scoring lexical** integrado en Elasticsearch. Con la configuración predeterminada, Elasticsearch usa BM25 para puntuar resultados de búsquedas de texto, salvo que configures otra similitud.

Lo mencionamos porque más adelante verás valores `_score` y necesitamos responder:

> ¿por qué un documento aparece antes que otro?

BM25 usa, entre otras cosas:

1. **frecuencia del término** dentro del documento;
2. **rareza del término en el corpus**;
3. **longitud del documento**;
4. **saturación**: repetir una palabra muchas veces aporta cada vez menos.

### ¿Qué significa lexical?

Basado principalmente en los **términos** presentes en consulta y documentos. No significa que el motor comprenda el significado como lo haría una búsqueda semántica.

### ¿Se instala?

- **En Elasticsearch:** no. BM25 ya está integrado y es la similitud predeterminada.
- **En Python:** existen paquetes llamados `rank-bm25`, pero son implementaciones locales distintas. **S07 ya no los necesita** para enseñar Elasticsearch.

Después de indexar el corpus ejecutaremos una consulta real en Elasticsearch y observaremos `_score`.

### C2. Por qué `contains()` no reemplaza a un buscador

Supón:

```text
D1 = "Mantenimiento de aeronaves KFIR"
D2 = "Mantenimiento preventivo de vehículos"
```

Para la palabra `mantenimiento`:

```text
D1 → True
D2 → True
```

Pandas puede decir que ambos contienen la cadena, pero ese `True/False` no responde cuál documento atiende mejor la consulta `mantenimiento aeronaves`.

Elasticsearch añade otras piezas:

**análisis del texto → índice invertido → recuperación → scoring BM25 → ranking**.

No ejecutamos código en esta celda porque BM25 será observado directamente en Elasticsearch, no mediante un simulador Python aparte.

# D · Console antes de Python

Console es donde probamos la API directamente antes de automatizarla.

## D0. Abrir Console

1. Entra a tu proyecto Elastic.
2. Menú o buscador global → **Dev Tools**.
3. Abre **Console**.
4. Pega una solicitud en el panel izquierdo.
5. Pulsa ▶.
6. Lee la respuesta en el panel derecho.

## D1. Probar `_analyze` sin crear un índice

**ELASTIC CONSOLE · NO SE EJECUTA EN COLAB**

```text
POST /_analyze
{
  "analyzer": "spanish",
  "text": "Servicios de mantenimiento de las aeronaves"
}
```

Esto funciona con el analyzer incorporado `spanish`; todavía no necesitas que exista `s07-demo`.

**Qué debes mirar:** `tokens[]`, `token`, `position`, `start_offset`, `end_offset`.

## D2. Crear un índice demo

```text
PUT /s07-demo
{
  "mappings": {
    "properties": {
      "titulo": {"type": "text", "analyzer": "spanish"},
      "categoria": {"type": "keyword"}
    }
  }
}
```

**Debe aparecer:** `acknowledged: true`.

## D3. Analizar en el contexto de un índice

Ahora que `s07-demo` existe, también puedes usar:

```text
POST /s07-demo/_analyze
{
  "analyzer": "spanish",
  "text": "Servicios de mantenimiento de las aeronaves"
}
```

La diferencia pedagógica es importante: **`/_analyze` funciona antes de crear el índice; `/s07-demo/_analyze` usa una ruta asociada a un índice que ya debe existir.**

# E · Conectar Python con Elasticsearch real

Hasta aquí ya usaste Console directamente. Ahora vas a conectar Colab al mismo motor.

## E0. Crear o abrir tu proyecto Elasticsearch

Para esta sesión no necesitas instalar Elasticsearch localmente, Docker, Bash ni una máquina virtual.

1. Entra a Elastic Cloud con tu cuenta.
2. Abre un proyecto Elasticsearch existente o crea uno.
3. Espera hasta poder entrar al proyecto.
4. Dentro del proyecto localiza la **Project URL / Elasticsearch endpoint**.
5. Localiza la opción de **API keys** del proyecto y crea una clave con permisos suficientes para la práctica.
6. Guarda temporalmente URL y API key: las introducirás en Colab.

La posición exacta de estos botones puede cambiar. Si no ves “API keys” o “Dev Tools”, usa el buscador global del proyecto.

### ¿Qué es la Project URL?

Es la dirección HTTPS del servicio Elasticsearch. Le dice al cliente Python **a dónde** enviar las solicitudes.

Ejemplo conceptual:

```text
https://mi-proyecto....elastic.cloud
```

No es una contraseña.

### ¿Qué es una API key?

Es una credencial que autoriza solicitudes. En esta clase la usamos para que Python pueda hablar con tu proyecto.

**Regla de seguridad:** no escribas la API key directamente en una celda, no la subas a GitHub y no la dejes visible en una captura. El siguiente bloque usa `getpass()` para que no quede impresa.

## E1. Qué vamos a comprobar

Crearemos:

```python
client = Elasticsearch(endpoint, api_key=api_key)
```

y después ejecutaremos:

```python
client.info()
```

Si `client.info()` responde, tenemos evidencia de que URL + credencial + red permiten hablar con Elasticsearch.

| Síntoma | Primera revisión |
|---|---|
| 401 / 403 | API key y permisos |
| timeout | Project URL, red o disponibilidad |
| URL inválida | debe incluir `https://` |
| 404 de índice | nombre del índice o todavía no fue creado |

**Qué NO significa:** `client` no contiene tus documentos; es solamente el objeto Python que sabe cómo enviar solicitudes al servicio.


In [ ]:
from getpass import getpass
from elasticsearch import Elasticsearch, helpers

endpoint = input('Project URL / endpoint: ').strip()
api_key = getpass('API key: ').strip()

client = Elasticsearch(endpoint, api_key=api_key)
info = client.info()
print('Conexión verificada con:', info.get('cluster_name') or info.get('name'))

## E1. Índice personal y mapping

Cada estudiante/equipo usa un índice propio para no pisar el trabajo de los demás.

**Decisiones de mapping:**

- `nombre_proceso` y `descripcion`: `text` con analyzer `spanish`;
- `id_proceso`, `entidad`, `tipo_registro`: `keyword`;
- `url_secop`: `keyword`.

In [ ]:
ALIAS = input('Alias corto sin espacios: ').strip().lower() or 'demo'
ALIAS = re.sub(r'[^a-z0-9_-]+','-', ALIAS)
INDEX_NAME = f's07-compras-claras-{ALIAS}'

mappings = {
    'properties': {
        'id_proceso': {'type':'keyword'},
        'nombre_proceso': {'type':'text', 'analyzer':'spanish'},
        'descripcion': {'type':'text', 'analyzer':'spanish'},
        'entidad': {'type':'keyword'},
        'tipo_registro': {'type':'keyword'},
        'url_secop': {'type':'keyword'}
    }
}

if client.indices.exists(index=INDEX_NAME):
    print('El índice ya existe:', INDEX_NAME)
else:
    client.indices.create(index=INDEX_NAME, mappings=mappings)
    print('Índice creado:', INDEX_NAME)

In [ ]:
resp = client.indices.analyze(
    index=INDEX_NAME,
    analyzer='spanish',
    text='Servicios de mantenimiento de las aeronaves'
)
print([t['token'] for t in resp['tokens']])

# F · Ingesta con bulk y verificación

**Qué queremos hacer:** pasar de tres documentos manuales a 1.994 procesos.

Cada acción `bulk` tiene tres piezas esenciales:

- `_index`: a dónde va;
- `_id`: identificador estable;
- `_source`: documento JSON que se guarda.

**Regla profesional:** ejecutar no basta; debes verificar `errors` y `count()`.

In [ ]:
def acciones_bulk(df):
    for _, r in df.iterrows():
        yield {
            '_index': INDEX_NAME,
            '_id': str(r['id_proceso']),
            '_source': {
                'id_proceso': str(r['id_proceso']),
                'nombre_proceso': str(r['nombre_proceso']),
                'descripcion': str(r['descripcion']),
                'entidad': str(r['entidad']),
                'tipo_registro': str(r['tipo_registro']),
                'url_secop': str(r['url_secop'])
            }
        }

ok, errors = helpers.bulk(client, acciones_bulk(corpus), raise_on_error=False)
remote_count = client.count(index=INDEX_NAME)['count']
print('Acciones OK:', ok)
print('Errores bulk:', len(errors))
print('Conteo local:', len(corpus))
print('Conteo Elasticsearch:', remote_count)
print('¿Coinciden?:', len(corpus) == remote_count)

# G · Query DSL explicado

Construiremos la consulta por capas:

1. `match`: búsqueda textual en un campo;
2. `multi_match`: evidencia en varios campos;
3. boost: hipótesis de peso por campo;
4. `filter`: condición exacta que no suma score;
5. `highlight`: inspección de fragmentos coincidentes.

## G1. `match` vs `term`

- `match` analiza texto y sirve para full-text.
- `term` busca un valor exacto y sirve para `keyword`, IDs o categorías.

No uses `term` sobre un campo `text` esperando comportamiento de búsqueda textual.

In [ ]:
resp_match = client.search(
    index=INDEX_NAME,
    size=5,
    query={'match': {'descripcion': consulta}}
)

for h in resp_match['hits']['hits']:
    print(round(h['_score'],3), h['_id'], h['_source']['nombre_proceso'][:100])

## G2. `multi_match`, `filter` y `highlight`

La consulta siguiente dice:

- busca `mantenimiento aeronaves`;
- dale más peso a `nombre_proceso`;
- usa `descripcion` como evidencia secundaria;
- restringe a `historico_adjudicado`;
- devuelve fragmentos para inspección humana.

In [ ]:
query_B = {
    'bool': {
        'must': [{
            'multi_match': {
                'query': consulta,
                'fields': ['nombre_proceso^3','descripcion']
            }
        }],
        'filter': [{'term': {'tipo_registro': 'historico_adjudicado'}}]
    }
}

resp_B = client.search(
    index=INDEX_NAME,
    size=5,
    query=query_B,
    highlight={'fields': {'nombre_proceso': {}, 'descripcion': {}}}
)

rows=[]
for rank,h in enumerate(resp_B['hits']['hits'], start=1):
    rows.append({
        'rank': rank,
        'id_proceso': h['_id'],
        'score': h['_score'],
        'nombre_proceso': h['_source']['nombre_proceso'],
        'highlight': str(h.get('highlight',{}))[:250]
    })
tabla_B = pd.DataFrame(rows)
display(tabla_B)

# H · Evaluar relevancia con Precision@5

Antes de etiquetar, define un criterio observable.

Ejemplo: “relevante = el documento trata directamente mantenimiento de aeronaves, no solo menciona una aeronave de forma tangencial”.

Después etiqueta top 5 con 1/0.

`Precision@5 = relevantes en los cinco primeros / 5`

Esta lógica escala profesionalmente con `_rank_eval`, pero aquí la hacemos manual para entenderla.

In [ ]:
criterio = input('Criterio observable de relevancia: ').strip()
print('Criterio:', criterio)

etiquetas = []
for _, row in tabla_B.iterrows():
    print('
Rank', row['rank'], '|', row['nombre_proceso'][:160])
    val = input('¿Relevante? 1=sí, 0=no: ').strip()
    etiquetas.append(1 if val == '1' else 0)

p5 = sum(etiquetas) / 5
print('Precision@5:', p5)

# I · Reto de transferencia: Sala de redacción

Ahora cambia el dominio. Una editora busca antecedentes sobre **sobrecostos en contratación**.

Tendrás campos como:

- `titulo`;
- `subtitulo`;
- `cuerpo`;
- `categoria`;
- `premium`;
- `publicado`.

Decisiones esperadas:

- texto largo → `text`;
- categoría → `keyword`;
- premium → `boolean`;
- publicado → `date`;
- A: `titulo + subtitulo + cuerpo`;
- B: `titulo^4 + subtitulo^2 + cuerpo` + filtro `premium=false`.

No basta copiar: debes justificar mapping, query y evaluación.

In [ ]:
URL_NOTICIAS = 'https://raw.githubusercontent.com/jazaineam1/BigData2026/main/Datos/noticias_contratacion_2026.json'
try:
    noticias = pd.read_json(URL_NOTICIAS)
    print('Noticias cargadas:', len(noticias))
    display(noticias.head(3))
except Exception as e:
    print('No se pudo cargar el dataset de noticias desde la URL:', e)
    print('El reto puede completarse después de verificar la ruta del archivo.')

# J · Exportación final

La entrega debe demostrar proceso, no solo resultados. Incluye:

1. índice usado;
2. consulta;
3. top 5;
4. criterio de relevancia;
5. P@5;
6. falso positivo o resultado dudoso;
7. siguiente experimento.

In [ ]:
salida = {
    'index': INDEX_NAME,
    'consulta': consulta,
    'criterio': criterio,
    'precision_at_5': p5,
    'top5': tabla_B.to_dict(orient='records')
}

Path('s07_config_busqueda.json').write_text(json.dumps(salida, ensure_ascii=False, indent=2), encoding='utf-8')
tabla_B.to_csv('s07_resultados_busqueda.csv', index=False)

md = f'''# Hito S07 · Relevancia textual

- Índice: {INDEX_NAME}
- Consulta: {consulta}
- Criterio: {criterio}
- Precision@5: {p5}

## Límite
La relevancia textual no demuestra irregularidad, calidad contractual ni riesgo. Solo ordena documentos según la consulta y configuración usadas.
'''
Path('hito_s07_relevancia.md').write_text(md, encoding='utf-8')
print('Archivos creados: s07_config_busqueda.json, s07_resultados_busqueda.csv, hito_s07_relevancia.md')